In [ ]:
#!/usr/bin/env python3
"""
Enrich an accident dataset with historical accident-count features.

What it does
------------
1) Reads the source CSV.
2) Deduplicates to one row per collision_id for the counting step.
3) Builds 9 features:
   - radii: 50m, 100m, 200m
   - lookback windows: 1, 2, 3 years
4) Counts how many *previous* accidents happened within each distance/time window.
5) Merges the features back to the original table so every casualty row gets the
   same accident-history features as its collision_id.

Why these windows?
------------------
On this file, these windows give a good spread between:
- very local repetition (50m),
- street/block-level repetition (100m),
- neighborhood-level repetition (200m),

while 1/2/3 years provide short-, medium-, and longer-term history without
becoming too broad.

Output column names
-------------------
past_accidents_50m_1y
past_accidents_50m_2y
past_accidents_50m_3y
past_accidents_100m_1y
...
past_accidents_200m_3y
"""

from __future__ import annotations

import argparse
import math
import re
from collections import defaultdict, deque
from pathlib import Path
from typing import Deque, Dict, Tuple

import numpy as np
import pandas as pd


RADIUS_LIST_M = [50, 100, 200, 300, 400, 500]
YEARS_LIST = [1,2,3, 4, 5]
CELL_SIZE_M = 100.0  # grid cell size for fast local searches


def parse_accident_datetime(df: pd.DataFrame) -> pd.Series:
    """Parse a robust accident datetime from accidentdate + time.

    Falls back to date-only when time is missing or malformed.
    """
    accident_date = pd.to_datetime(df["accidentdate"], errors="coerce", dayfirst=True)

    # Build a combined string where possible, then fallback to date only.
    time_str = df["time"].astype("string").str.strip()
    combined = accident_date.dt.strftime("%Y-%m-%d") + " " + time_str
    dt = pd.to_datetime(combined, errors="coerce")
    dt = dt.fillna(accident_date)
    return dt


def cell_of(x: float, y: float, cell_size: float) -> Tuple[int, int]:
    return (int(math.floor(x / cell_size)), int(math.floor(y / cell_size)))


def count_past_accidents(
    accidents: pd.DataFrame,
    years_list: list[int] = YEARS_LIST,
    radii_list_m: list[int] = RADIUS_LIST_M,
    cell_size_m: float = CELL_SIZE_M,
    severity_list: list[str] = ["Slight", "Serious", "Fatal"],
) -> pd.DataFrame:
    """Count prior accidents within each (year, radius, severity) combination.

    accidents must already be deduplicated to one row per collision_id and
    sorted chronologically by `dt`.
    """
    accidents = accidents.copy()

    max_years = max(years_list)
    max_radius = max(radii_list_m)
    max_radius_sq = max_radius * max_radius
    r_cells = int(math.ceil(max_radius / cell_size_m))

    severity_to_idx = {s: i for i, s in enumerate(severity_list)}

    # Store only recent points needed for the maximum lookback window.
    grid: Dict[Tuple[int, int], Deque[Tuple[pd.Timestamp, float, float, int]]] = defaultdict(deque)

    # All severities combined
    counts = np.zeros(
        (len(accidents), len(years_list), len(radii_list_m)),
        dtype=np.int32,
    )

    # Severity-specific counts
    severity_counts = np.zeros(
        (len(accidents), len(years_list), len(radii_list_m), len(severity_list)),
        dtype=np.int32,
    )

    # Local references for speed.
    years_list = list(years_list)
    radii_sq = [r * r for r in radii_list_m]

    for i, row in enumerate(accidents.itertuples(index=False)):
        dt = row.dt
        x = float(row.easting)
        y = float(row.northing)

        max_cutoff = dt - pd.DateOffset(years=max_years)

        cx, cy = cell_of(x, y, cell_size_m)

        # Scan only nearby grid cells.
        for ix in range(cx - r_cells, cx + r_cells + 1):
            for iy in range(cy - r_cells, cy + r_cells + 1):
                dq = grid.get((ix, iy))
                if not dq:
                    continue

                # Remove points older than the largest lookback window.
                while dq and dq[0][0] < max_cutoff:
                    dq.popleft()

                for past_dt, px, py, past_sev_idx in dq:
                  if past_dt >= dt:
                      continue

                  dx = x - px
                  dy = y - py
                  d2 = dx * dx + dy * dy

                  for yi, years in enumerate(years_list):
                      cutoff = dt - pd.DateOffset(years=years)
                      if past_dt < cutoff:
                          continue

                      for ri, r2 in enumerate(radii_sq):
                          if d2 <= r2:
                              counts[i, yi, ri] += 1
                              severity_counts[i, yi, ri, past_sev_idx] += 1
        # Add the current accident after counting so it cannot count itself.
        current_sev_idx = severity_to_idx.get(row.collision_severity, 0)
        grid[(cx, cy)].append((dt, x, y, current_sev_idx))

    out = accidents[["collision_id"]].copy()
    out = out.set_index("collision_id")

    for yi, years in enumerate(years_list):
        for ri, radius_m in enumerate(radii_list_m):

            # All severities combined
            col = f"past_accidents_{radius_m}m_{years}y"
            out[col] = counts[:, yi, ri]

            # Severity-specific
            for si, sev in enumerate(severity_list):
                sev_col = f"past_accidents_{radius_m}m_{years}y_{sev.lower()}"
                out[sev_col] = severity_counts[:, yi, ri, si]
    out = out.reset_index()
    return out

def main() -> None:
    input_path = "all_59k_accidents_no_history.csv"
    output_path = "all_59k_accidents_complete_final_cleaned.csv"

    df = pd.read_csv(input_path, low_memory=False)

    if "collision_id" not in df.columns:
        raise ValueError("Missing collision_id")
    if "accidentdate" not in df.columns or "time" not in df.columns:
        raise ValueError("Missing accidentdate/time")
    if "easting" not in df.columns or "northing" not in df.columns:
        raise ValueError("Missing coordinates")

    severity_order = {"Slight": 1, "Serious": 2, "Fatal": 3}

    accidents = df[
        ["collision_id", "accidentdate", "time", "easting", "northing", "casualty_severity"]
    ].copy()

    accidents["_sev_rank"] = (
        accidents["casualty_severity"].map(severity_order).fillna(0).astype(int)
    )

    accidents = (
        accidents.sort_values(["collision_id", "_sev_rank"])
        .drop_duplicates("collision_id", keep="last")
        .drop(columns=["_sev_rank"])
    )

    accidents["dt"] = parse_accident_datetime(accidents)
    accidents = accidents.dropna(subset=["dt"])
    accidents = accidents.sort_values(["dt", "collision_id"]).reset_index(drop=True)

    accidents = accidents.rename(columns={"casualty_severity": "collision_severity"})

    features = count_past_accidents(accidents)

    enriched = df.merge(features, on="collision_id", how="left")

    count_cols = [c for c in enriched.columns if c.startswith("past_accidents_")]
    enriched[count_cols] = enriched[count_cols].fillna(0).astype("int32")

    enriched.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")


main()